In [1]:
import os
import json
import pandas as pd
from cog_analysis import load_boat_data
from report_fct import filter_interval
import numpy as np
import os.path as osp

def build_csv_from_summary(summary_path, data_root, output_csv="all_data.csv"):
    with open(summary_path, "r") as f:
        summary = json.load(f)

    all_rows = []

    for run_entry in summary:
        run_name = run_entry["run"]
        intervals = run_entry["intervals"]

        date_part = run_name.split("_Run")[0]
        run_path = osp.join(data_root, date_part, run_name)
        csv_files = [f for f in os.listdir(run_path) if f.endswith(".csv")]

        csv_paths = [os.path.join(run_path, f) for f in csv_files]

        df1, name1 = load_boat_data(csv_paths[0])

        for i, interval in enumerate(intervals):
            start, end = interval["start_time"], interval["end_time"]
            if end - start < 30:
                print(f"Skipping interval {i + 1} for {run_name}: duration < 30 seconds")
                continue

            df1_clip = filter_interval(df1, start, end)
            if df1_clip.empty:
                print(f"Skipping interval {i + 1} for {run_name}: no data in interval")
                continue
            if interval["boat1_master_leeward"]:
                master_df = df1_clip
            else:
                slave_df = df1_clip
            for df_clip, prefix, other_prefix in [(df1_clip, "boat1", "boat2")]:
                df = df_clip.copy()
                df["run"] = run_name
                df["interval_id"] = i + 1
                df["boat_name"] = interval.get(f"{prefix}_name", "")
                df["opponent_name"] = interval.get(f"{other_prefix}_name", "")
                df["boat_role"] = "master" if interval.get(f"{prefix}_master_leeward", False) else "slave"
                df["boat_weight"] = interval.get(f"{prefix}_total_weight", None)
                df["interval_duration"] = interval.get("duration", None)
                df["mast_brand"] = interval.get(f"{prefix}_mast_brand", None)
                df["load_cell"] = interval.get(f"{prefix}_load_cell_availability", None)
                df["pol_ratio"] = interval.get(f"{prefix}_pol_ratio", None)
                df["SOG_ref"]   = interval.get(f"{prefix}_SOG_ref", None)
                df["leg_type"]   = interval.get("leg_type", None)
                gain_forward = []
                gain_lateral = []
                gain_vmg = []

                # for t in df["SecondsSince1970"]:
                #     m_clip = master_df[master_df["SecondsSince1970"] <= t]
                #     s_clip = slave_df[slave_df["SecondsSince1970"] <= t]
                #     gain_df = compute_directional_gain(m_clip, s_clip)

                #     if gain_df.empty:
                #         gain_forward.append(np.nan)
                #         gain_lateral.append(np.nan)
                #         gain_vmg.append(np.nan)
                #     else:
                #         gain_forward.append(gain_df.loc["Total Gain", "Forward"])
                #         gain_lateral.append(gain_df.loc["Total Gain", "Lateral"])
                #         gain_vmg.append(gain_df.loc["Total Gain", "VMG"])

                # # === assign raw gains (master reference frame) ===
                # df["gain_forward"] = gain_forward
                # df["gain_lateral"] = gain_lateral
                # df["gain_vmg"]     = gain_vmg

                # # === normalize to boat reference frame ===
                # sign = 1 if df["boat_role"].iloc[0] == "master" else -1
                # df["gain_forward"] *= sign
                # df["gain_lateral"] *= sign
                # df["gain_vmg"]     *= sign

                lines = df[["lateral", "central"]].values
                sorted_lines = np.sort(lines, axis=1)

                # Simple explicit assignment
                # df["Line_R2"] = df["lateral"]
                # df["Line_L2"] = df["lateral"]   # not interesting anymore be we keep this for uniformity with port camargue
                # df["Line_C2"] = df["central"]
                # df["side_line2"] = df["Line_R2"] + df["Line_L2"]
                # df["total_line2"] = df["side_line2"] + df["Line_C2"]
                all_rows.append(df)

    df_global = pd.concat(all_rows, ignore_index=True)
    df_global.to_csv(output_csv, index=False)
    print(f"✅ Global CSV saved to: {output_csv}")



In [2]:
build_csv_from_summary(
    summary_path="summary_enriched.json",
    data_root= "../Data_Sailnjord/Hyères November 2025/Straight_lines",
    output_csv="all_data.csv"
)

✅ Global CSV saved to: all_data.csv


In [3]:

import pandas as pd
df_global = pd.read_csv("all_data.csv")
print(df_global.columns.tolist())

['ISODateTimeUTC', 'SecondsSince1970', 'SOG', 'TWA', 'TWS', 'TWD', 'BelowLineCalc', 'central', 'COG', 'COG_Mag', 'DistanceToLeader', 'Heel', 'Heel_Abs', 'Heel_Lwd', 'Lat', 'LatBow', 'LatCenter', 'lateral', 'LatStern', 'Leg', 'Log', 'LogAlongCourse', 'Lon', 'LonBow', 'LonCenter', 'LonStern', 'MagneticVariation', 'Rank', 'ROT', 'TimeLocal', 'TimeUTC', 'Trim', 'TWA_Abs', 'VMC', 'VMG', 'XTE', 'run', 'interval_id', 'boat_name', 'opponent_name', 'boat_role', 'boat_weight', 'interval_duration', 'mast_brand', 'load_cell', 'pol_ratio', 'SOG_ref', 'leg_type']
